<a href="https://colab.research.google.com/github/netsetos/agentic-ai-weekend-gcp-learners/blob/rag-production-hardening/module-13-capstone-defend-documind/lesson-13.3-capstone-hardening/notebooks/GCP_Capstone_13.3_CapstoneHardening.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 13.3 — Build Phase 2: Break It on Purpose, Then Price It

A failure-modes document listing only the failures you handle is a marketing page. This phase makes you cause four, write down which ones your monitoring missed, and turn the bill into a model that answers what breaks first at ten times the volume.


## Cell 1: Four Failures, Each With a Script

Not four paragraphs. Four things you can run.


In [ ]:
# Four failure modes, each proved by a script. Not four paragraphs.
#
# The rubric's 2 for failure modes is "four documented modes with real symptoms,
# each reproduced by a script". The 3 adds: which ones your monitoring catches.
# Every 'how' below is a command the kit already has; FAILURE-MODES.md wraps each
# in a script of yours, and the alerts named are the ones alerts.tf declares (12.3).
FAILURES = [
    ('redeliver a stale generation of a document',
     'gcloud pubsub topics publish documind-ingest --message <the object record with generation - 1> (12.5, Cell 8)',
     'nothing changes: the worker acks it as ingest_stale_event; the ledger, the chunks and the answers stay where they were',
     'caught: documind/ingest_events counts it by event (12.3); nothing pages, by design - a late redelivery is normal'),
    ('expire the context cache',
     'client.caches.delete(name=...) on the tenant pack; make cache TENANT= rebuilds it (10.2)',
     'the next answers pay full price for the pack: cached_tokens 0, cost_usd up ~10x, latency up',
     'caught: the p95 alert (12.3), if the slow answers last five minutes - the cost is not'),
    ('take the gateway away from a gateway-backed candidate',
     'make candidate MODEL_BACKEND=gateway, then make gateway-off (11.3)',
     '502 from the generator on the candidate; no usage row is written at all',
     'NOT caught - the alerts read rows and latency, and a 502 leaves neither'),
    ('poison the ingest queue',
     'make poison (12.5): a zero-byte object under the tenant prefix',
     'the worker answers 400, logs ingest_poison, the message lands in ingest-dlq-sub minutes later; nothing else changes',
     'NOT caught - nothing watches DLQ depth; make dlq is a person looking'),
]
print(f'  {"what you break":40} {"symptom":60} monitoring')
for what, how, symptom, caught in FAILURES:
    print(f'  {what:40} {symptom:60} {caught}')
    print(f'  {"":40} how: {how}')
print()
print('  Two of the four are NOT caught, and writing that down is worth more')
print('  (the first is caught and does not page: a metric that counts a non-event')
print('  is a different thing from an alarm, and the difference is a decision)')
print('  than fixing one of them. A failure-modes document listing only the')
print('  failures you handle is a marketing page.')
print()
print('  The rubric asks what you did about the uncaught group. "Added a DLQ')
print('  depth alert" is one answer. "Accepted it, because the DLQ is checked')
print('  daily and a day of stale documents is survivable" is an equally good')
print('  one - it is a decision, not an oversight.')

## Cell 2: The Isolation Test That Can Actually Fail

Not must_not_contain. A status code, before a model is involved at all.


In [ ]:
# The isolation assertion that goes in YOUR pipeline.
#
# Red first. An assertion you have only ever seen green is a hypothesis.
def enforce_membership(email: str, tenant: str, roster: dict, loosened: bool) -> int:
    """Returns the HTTP status. `loosened` is the one-line regression."""
    if loosened:
        return 200                      # somebody 'fixed' a support ticket
    return 200 if email in roster.get(tenant, set()) else 403


ROSTER = {'acme': {'priya@acme.in'}, 'zeta': {'ravi@zeta.in'}}
TRIES = [
    ('a member asking their own tenant', 'priya@acme.in', 'acme'),
    ('a member asking ANOTHER tenant',   'priya@acme.in', 'zeta'),
    ('a stranger',                       'mallory@evil.in', 'acme'),
]
for loosened in (False, True):
    label = 'AFTER somebody loosens the check' if loosened else 'as shipped'
    print(f'  {label}:')
    for what, who, tenant in TRIES:
        code = enforce_membership(who, tenant, ROSTER, loosened)
        ok = code == 403 or what.endswith('own tenant')
        print(f'      {code}  {what:34} {"expected" if ok else "<-- LEAK"}')
    print()
print('  The right-hand column is your CI assertion: rows 2 and 3 must be 403.')
print('  Not "must not contain the other tenant\'s figure" - 403, on the status')
print('  line, before a model is involved at all.')
print()
print('  Prove it red before you trust it. Set loosened=True, watch the pipeline')
print('  fail, put it back. That red run is what the rubric asks you to point at.')

## Cell 3: What It Costs, and What Breaks First

Scaling the bill by ten is a 2. Naming the thing that stops being linear is a 3.


In [ ]:
# COST-MODEL.md, computed. Three volumes, both price regimes, in rupees.
USD_INR = 85
# Per 1M tokens. Intro runs to 31 Dec 2026; standard from 1 Jan 2027.
PRICES = {'intro': (0.75, 3.75), 'standard': (1.50, 7.50)}
TOKENS_IN, TOKENS_OUT = 2400, 350          # measure YOUR OWN: make usage HOURS=24 prints them per model (12.3); these are DocuMind's, uncached
RETRIEVAL_PER_QUERY = 0.00012              # a Firestore vector query + the chunk reads; the Vector Search endpoint is a per-hour line, not a per-query one
HOSTING_PER_MONTH = 0.0                    # Cloud Run at 0 min-instances; an L4 kept warm is Rs 86,904 a month (11.4) - a rate, not a price per answer


def monthly(queries: int, regime: str) -> dict:
    pin, pout = PRICES[regime]
    model = queries * (TOKENS_IN * pin + TOKENS_OUT * pout) / 1_000_000
    retrieval = queries * RETRIEVAL_PER_QUERY
    total = model + retrieval + HOSTING_PER_MONTH
    return {'model': model, 'retrieval': retrieval, 'hosting': HOSTING_PER_MONTH,
            'total_usd': total, 'total_inr': total * USD_INR,
            'per_answer_inr': total * USD_INR / queries}


print(f'  {"queries/mo":>11}  {"regime":9} {"model":>9} {"retr":>8} '
      f'{"TOTAL Rs":>11} {"per answer":>11}')
for q in (1_000, 10_000, 100_000):
    for regime in ('intro', 'standard'):
        c = monthly(q, regime)
        print(f'  {q:>11,}  {regime:9} ${c["model"]:>8.2f} ${c["retrieval"]:>7.2f} '
              f'Rs {c["total_inr"]:>8,.0f} Rs {c["per_answer_inr"]:>8.2f}')
print()
print('  Two regimes on every row, with the switch date, because a single-priced')
print('  table is out of date on 1 January and nobody notices.')
print()
print('  make compare (11.4) prices one question through every backend on the lane;')
print('  put ITS numbers in COST-MODEL.md, not typed ones.')
print()
print('  Now the part that separates a 2 from a 3: WHAT BREAKS FIRST at 10x?')
BREAKS = [
    ('100k queries/mo', 'Cloud Run concurrency 8 -> more instances -> cold starts on the tail'),
    ('~40k chunks',     'Firestore p95 goes from 180ms to ~1.9s - the day the Vector Search endpoint earns the bill it has run up since make up'),
    ('per-tenant caches', 'each is 4,096+ tokens minimum and bills hourly whether used or not'),
    ('Vector Search',   'min_replica_count=1 is a floor, not a scale - it is the SECOND replica'),
    ('the SLM lane',    'one L4 by design (make gpu-cap holds the quota at 1); past it the answer is GKE (11.5), not a bigger number'),
]
for when, what in BREAKS:
    print(f'  {when:20} {what}')
print()
print('  None of those is the bill going up tenfold. That is the answer the')
print('  rubric is listening for: at 10x something stops being LINEAR.')

---

The rubric and the eight components are published in `course-bibles/capstone-rubric.md`. Score yourself against them before somebody else does.
